# Tutorial 4: Conversation-Level Analysis

This notebook teaches you how to analyze multi-turn conversations — tracing how information flows across turns, which turns the model attends to, and how earlier context influences later responses.

**What you'll learn:**
- Turn detection from chat templates
- `Turn` and `TurnList` for navigating conversations
- `ConversationTrace` for per-turn activation analysis
- Cross-turn attention patterns
- Practical examples: name recall, fact propagation

**Prerequisites:** A model with a tokenizer and chat template (e.g., Llama 3 Instruct).

## 1. Setup

In [ ]:
from mlxterp import InterpretableModel
from mlxterp.conversation import Turn, TurnList, detect_turns
import mlx.core as mx

model = InterpretableModel("mlx-community/Llama-3.2-1B-Instruct-4bit")
print(f"Model: {len(model.layers)} layers")
print(f"Tokenizer: {type(model.tokenizer).__name__}")
print(f"Has chat template: {hasattr(model.tokenizer, 'apply_chat_template')}")

## 2. Understanding Turns

Chat models use special tokens to separate turns. When you send a conversation like:

```
User: My name is Alice.
Assistant: Nice to meet you!
User: What's my name?
```

The tokenizer wraps each turn with template tokens (role markers, end-of-turn markers). mlxterp detects these boundaries automatically.

In [ ]:
# Define a conversation
conversation = [
    {"role": "user", "content": "My name is Alice."},
    {"role": "assistant", "content": "Nice to meet you, Alice!"},
    {"role": "user", "content": "What is my name?"},
]

# Detect turn boundaries
turns = detect_turns(model.tokenizer, conversation)

print(f"Detected {len(turns)} turns:")
print(turns)

In [ ]:
# Examine each turn in detail
for turn in turns:
    print(f"\nTurn {turn.index} ({turn.role}):")
    print(f"  Content tokens: positions {turn.content_start} to {turn.content_end}")
    print(f"  Full turn:      positions {turn.full_start} to {turn.full_end}")
    print(f"  Content token count: {turn.n_content_tokens}")
    print(f"  Total token count:   {turn.n_total_tokens}")

In [ ]:
# See the actual tokens with turn boundaries marked
full_tokens = model.tokenizer.apply_chat_template(conversation, tokenize=True, add_generation_prompt=False)

print("Full tokenized conversation:")
for i, tid in enumerate(full_tokens):
    token_str = model.tokenizer.decode([tid]).replace('\n', '\\n')
    
    # Mark turn boundaries
    markers = []
    for turn in turns:
        if i == turn.content_start:
            markers.append(f"← Turn {turn.index} ({turn.role}) content start")
        if i == turn.content_end:
            markers.append(f"← Turn {turn.index} content end")
    
    marker_str = "  " + ", ".join(markers) if markers else ""
    print(f"  [{i:3d}] {token_str:20s}{marker_str}")

## 3. TurnList: Filtering and Navigating

The `TurnList` class provides convenient methods for working with turns.

In [ ]:
# Indexing
print(f"First turn: {turns[0].role} — '{conversation[0]['content']}'")
print(f"Last turn: {turns[-1].role} — '{conversation[-1]['content']}'")

# Slicing
first_two = turns[0:2]
print(f"\nFirst two turns: {len(first_two)} turns")
print(f"  Roles: {first_two.roles}")

# Filter by role
user_turns = turns.by_role("user")
assistant_turns = turns.by_role("assistant")
print(f"\nUser turns: {len(user_turns)}")
print(f"Assistant turns: {len(assistant_turns)}")

# All roles in order
print(f"\nRole sequence: {turns.roles}")

# Get all content positions
content_pos = turns.content_positions()
print(f"All content positions: {content_pos}")

## 4. ConversationTrace: Per-Turn Activation Analysis

The `ConversationTrace` context manager runs a single forward pass on the full conversation and then lets you slice activations by turn.

In [ ]:
with model.conversation_trace(conversation) as ct:
    print(f"Turns detected: {len(ct.turns)}")
    print(f"Total activations captured: {len(ct.activations)}")
    print(f"Output shape: {ct.output.shape}")
    print(f"\nTurn details:")
    for turn in ct.turns:
        print(f"  Turn {turn.index} ({turn.role}): tokens {turn.content_start}-{turn.content_end}")

In [ ]:
# Get activations for a specific turn
with model.conversation_trace(conversation) as ct:
    # Turn 0: "My name is Alice."
    turn0_act = ct.get_turn_activation(0, "layers.10")
    
    # Turn 2: "What is my name?" (the question)
    turn2_act = ct.get_turn_activation(2, "layers.10")
    
    if turn0_act is not None:
        print(f"Turn 0 (name mention) at layer 10: shape {turn0_act.shape}")
    if turn2_act is not None:
        print(f"Turn 2 (question) at layer 10: shape {turn2_act.shape}")

In [ ]:
# Compare content_only vs full turn
with model.conversation_trace(conversation) as ct:
    content_only = ct.get_turn_activation(0, "layers.5", content_only=True)
    full_turn = ct.get_turn_activation(0, "layers.5", content_only=False)
    
    if content_only is not None and full_turn is not None:
        print(f"Content only: {content_only.shape} (just message tokens)")
        print(f"Full turn:    {full_turn.shape} (includes template tokens)")

## 5. Cross-Turn Attention

One of the most powerful conversation-level analyses: **how much does each turn attend to other turns?**

The cross-turn attention matrix aggregates token-level attention into a turn-level summary.

In [ ]:
with model.conversation_trace(conversation) as ct:
    # Get cross-turn attention at a specific layer and head
    cross_attn = ct.cross_turn_attention(layer=10, head=0)
    
    if cross_attn is not None:
        mx.eval(cross_attn)
        print("Cross-turn attention matrix (layer 10, head 0):")
        print(f"Shape: {cross_attn.shape} ({len(ct.turns)} turns x {len(ct.turns)} turns)")
        print()
        
        # Pretty print
        roles = [t.role for t in ct.turns]
        header = "          " + "  ".join(f"{r[:5]:>8s}" for r in roles)
        print(header)
        for i, row in enumerate(cross_attn.tolist()):
            vals = "  ".join(f"{v:>8.4f}" for v in row)
            print(f"{roles[i]:>8s}  {vals}")
    else:
        print("No attention weights captured (model may not expose them)")

In [ ]:
# Scan across layers: where does the question attend to the name?
with model.conversation_trace(conversation) as ct:
    print("Cross-turn attention: Question (turn 2) → Name (turn 0)")
    print(f"{'Layer':>5}  {'Head 0':>8}  {'Head 1':>8}  {'Head 2':>8}  {'Head 3':>8}")
    print("-" * 50)
    
    for layer in range(0, len(model.layers), 2):  # Every other layer for speed
        scores = []
        for head in range(4):
            cross = ct.cross_turn_attention(layer=layer, head=head)
            if cross is not None:
                mx.eval(cross)
                scores.append(float(cross[2, 0]))  # Question → Name
            else:
                scores.append(0.0)
        
        vals = "  ".join(f"{s:>8.4f}" for s in scores)
        marker = "  ← high" if max(scores) > 0.1 else ""
        print(f"{layer:>5}  {vals}{marker}")

## 6. Example: Name Recall Analysis

Does the model truly "recall" the name from turn 0 when answering the question in turn 2?

In [ ]:
name_conversation = [
    {"role": "user", "content": "My name is Alice and I live in Paris."},
    {"role": "assistant", "content": "Nice to meet you, Alice! Paris is a beautiful city."},
    {"role": "user", "content": "What is my name and where do I live?"},
]

with model.conversation_trace(name_conversation) as ct:
    print(f"Turns: {len(ct.turns)}")
    for t in ct.turns:
        print(f"  Turn {t.index} ({t.role}): '{name_conversation[t.index]['content'][:40]}...'")
    
    print("\nAnalyzing where the question looks for information...")
    print()
    
    # Check multiple layers
    important_layers = []
    for layer in range(len(model.layers)):
        cross = ct.cross_turn_attention(layer=layer, head=0)
        if cross is not None:
            mx.eval(cross)
            # Question (turn 2) attending to user info (turn 0)
            score = float(cross[2, 0])
            if score > 0.05:
                important_layers.append((layer, score))
    
    if important_layers:
        print("Layers where question strongly attends to name/location info:")
        for layer, score in sorted(important_layers, key=lambda x: -x[1]):
            print(f"  Layer {layer}: attention = {score:.4f}")
    else:
        print("No strong cross-turn attention found at head 0")
        print("(Try checking other heads — the information may flow through different heads)")

## 7. Example: Fact Propagation

When a user states a fact in turn 1, does the model use it in turn 3?

In [ ]:
fact_conversation = [
    {"role": "user", "content": "The capital of France is Paris."},
    {"role": "assistant", "content": "That is correct."},
    {"role": "user", "content": "What is the capital of France?"},
]

with model.conversation_trace(fact_conversation) as ct:
    # Compare activations at the question turn across layers
    print("Activation norms at the question turn (last content token):")
    print(f"{'Layer':>5}  {'|activation|':>12}")
    print("-" * 20)
    
    for layer_idx in range(0, len(model.layers), 2):
        act = ct.get_turn_activation(2, f"layers.{layer_idx}")
        if act is not None:
            # Norm of last token
            last_token = act[0, -1, :].astype(mx.float32)
            norm = float(mx.sqrt(mx.sum(last_token * last_token)))
            print(f"{layer_idx:>5}  {norm:>12.2f}")

## 8. Converting to Results

Use `.to_result()` to get a structured `ConversationResult` for export.

In [ ]:
import json

with model.conversation_trace(conversation) as ct:
    result = ct.to_result()

print(result.summary())
print()
print("Turn metadata:")
for turn_info in result.turns:
    print(f"  {turn_info}")
print()

# JSON export
parsed = json.loads(result.to_json())
print(f"Result type: {parsed['result_type']}")
print(f"Data keys: {list(parsed['data'].keys())}")

## Summary

| Tool | Purpose | Key Method |
|------|---------|------------|
| `detect_turns()` | Find turn boundaries | `detect_turns(tokenizer, messages)` |
| `TurnList` | Navigate turns | `turns.by_role("user")`, `turns[0:2]` |
| `Turn` | Turn metadata | `.content_positions`, `.role`, `.n_content_tokens` |
| `ConversationTrace` | Multi-turn tracing | `model.conversation_trace(messages)` |
| `.get_turn_activation()` | Per-turn activations | `ct.get_turn_activation(turn_idx, component)` |
| `.cross_turn_attention()` | Turn-level attention | `ct.cross_turn_attention(layer, head)` |
| `.to_result()` | Export results | `ct.to_result().to_json()` |

**Key insight:** Conversation analysis lets you study how context accumulates across turns — something single-prompt analysis misses entirely. This is essential for understanding real chat model behavior.

---

**Congratulations!** You've completed the mlxterp tutorial series. You now know how to:
- Perform causal patching at layer/position/head level (Tutorial 1)
- Discover circuits with DLA, path patching, and ACDC (Tutorial 2)
- Generate text with interventions (Tutorial 3)
- Analyze multi-turn conversations (Tutorial 4)